# Foresight medallion -- SIT evidence run

Sprint 26 WS-A (issue #335). Deterministically materialises the Foresight-tier
Gold Delta tables that turn the descriptive occupancy surface into the
predictive tier (design spec 3.2 / D2):

- `gold.fact_occupancy_forecast` -- ward x horizon (0..72h) forecast occupancy.
- `gold.fact_forecast_driver`    -- the decomposition ('why') behind each point.
- `gold.fact_signal`             -- Trust-A projection over the Sprint 21
  `gold.ext_fact_signal` spine (deny-by-default), evidencing the seasonality driver.

Synthetic + deterministic, no PHI (ADR-0013 / ADR-0016). No LLM-guessed numbers
(design D2/D4). The transform logic lives in the sibling `build_gold_*.py`
notebook resources and is unit-tested offline under `tests/`.

**Dependency:** the `gold.fact_signal` cell needs the Sprint 21 external-signal
Gold tables (`gold.ext_fact_signal`, `gold.ext_dim_source`) already materialised
by `data-platform/notebooks/external-signals/run_ext_medallion.ipynb`.

In [ ]:
import sys

# Notebook resources (the sibling .py modules) live under builtin/ in Fabric.
if "builtin/foresight" not in sys.path:
    sys.path.insert(0, "builtin/foresight")

import build_gold_forecast as fc
import build_gold_signal as sig


In [ ]:
# Forecast + driver Gold tables from the deterministic DEFAULT_WARDS baseline.
# Swap DEFAULT_WARDS (or pass a real ward-series reader) here to plug in a real
# forecasting model -- the D2 seam. Contract + ontology binding stay unchanged.
fc.run()


In [ ]:
# Trust-A Foresight signal projection over the Sprint 21 external-signal spine.
sig.run()


In [ ]:
# Inline verification -- counts + the Medicine A 72h breach for the evidence doc.
for t in ["gold.fact_occupancy_forecast", "gold.fact_forecast_driver", "gold.fact_signal"]:
    print(t, spark.table(t).count())

display(
    spark.table("gold.fact_occupancy_forecast")
    .where("wardId = 'Medicine A' AND horizonH IN (0, 72)")
    .select("wardId", "horizonH", "forecastOccupiedBeds", "forecastOccupancyPct", "breach")
    .orderBy("horizonH")
)
